# C7-cnn-transfer — Practice p16 — Solution

The `layer4` opener expands immediately to the stage interface, so a single block is enough to emit 2048 channels and halve 14×14 to 7×7.

In [ ]:
# Cache pin (course convention, plan 009): pretrained weights live in the repo's
# gitignored reference/cache/ -- resolve it from the repo root BEFORE importing torch.
import os, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "pyproject.toml").exists())
os.environ["TORCH_HOME"] = str(_root / "reference" / "cache" / "torch")

import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

# float32 register (course exception): pretrained resnet50 is a float32 artifact.
# No float64 default here; inputs are cast .to(torch.float32) at the model
# boundary; repeat float32 forwards are bit-identical.
SEED = 20260804

model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
model.eval()
assert next(model.parameters()).dtype == torch.float32

torch.manual_seed(SEED)
x = torch.randn(2, 3, 224, 224).to(torch.float32)
ch = list(model.children())
trunk = nn.Sequential(*ch[:7], model.layer4[:1])
with torch.inference_mode():
    trunk_shape = tuple(trunk(x).shape)


In [ ]:
# Stem + layer1..3: 9,408 + 128 + 215,808 + 1,219,584 + 7,098,368.
hand_frozen = 8_543_296
# Five convs: 524,288 + 2,359,296 + 1,048,576 + 2,097,152;
# four BNs: 2*(512 + 512 + 2,048 + 2,048) = 10,240.
hand_thawed = 6_039_552


In [ ]:
for piece in ch[:7]:
    for p in piece.parameters():
        p.requires_grad = False
n_frozen = sum(p.numel() for p in trunk.parameters() if not p.requires_grad)
n_trainable = sum(p.numel() for p in trunk.parameters() if p.requires_grad)
counts_match = n_frozen == hand_frozen and n_trainable == hand_thawed
trainable_names = [name for name, p in trunk.named_parameters() if p.requires_grad]
trainable_prefix_ok = bool(trainable_names) and all(name.startswith("7.") for name in trainable_names)


The opener's `conv3` expands the residual branch to 2048 channels immediately.
Its `downsample` convolution and BatchNorm reshape the skip branch to the same
2048-channel, 7×7 interface, so the residual addition is defined.


### Answer check

In [ ]:
assert trunk_shape == (2, 2048, 7, 7)
assert (hand_frozen, hand_thawed) == (8_543_296, 6_039_552)
assert counts_match
assert trainable_prefix_ok
